## TO-DO
- fixation saccade data seems to be missing a lot after applying filter for fix_failed

In [1]:
import os
os.getcwd()

'c:\\Users\\Ailene\\OneDrive - California Institute of Technology\\Documents\\GitHub\\blindspot-multisensory\\Data'

Combine left and right eye behavioral data (.csv)

In [55]:
import os
import pandas as pd

# List all CSV files in the 'csv/' directory that start with 'SV' and end with '.csv'
csv_files = [f for f in os.listdir('csv/') if f.startswith('SV') and (f.endswith('_L.csv') or f.endswith('_R.csv'))]
print("CSV files found:", csv_files)

# Extract unique subject IDs by splitting filenames
sids = list(set([f.split('_')[0] for f in csv_files]))
print("Unique subject IDs:", sids)

# Combine Left (L) and Right (R) data for each subject
for sid in sids:

    if os.path.exists('csv/{}.csv'.format(sid)):
        print(f"Combined file for subject {sid} already exists. Skipping...")
        continue

    try:
        # Read left (L) and right (R) CSV files for the current subject ID
        df_L = pd.read_csv('csv/{}_L.csv'.format(sid))
        df_R = pd.read_csv('csv/{}_R.csv'.format(sid))
        
        # Combine the two DataFrames
        combined_df = pd.concat([df_L, df_R], ignore_index=True)
        # print(f"Combined data for subject {sid}:\n", combined_df)

        # Save the combined DataFrame to a new CSV file
        combined_df.to_csv('csv/{}.csv'.format(sid), index=False)
        print(f"Successfully combined {sid}_L.csv and {sid}_R.csv into {sid}.csv")
    
    except FileNotFoundError as e:
        print(f"Error: {e}. Skipping subject {sid}.")


CSV files found: ['SV009_L.csv', 'SV009_R.csv', 'SV012_L.csv', 'SV012_R.csv', 'SV013_L.csv', 'SV013_R.csv', 'SV017_L.csv', 'SV017_R.csv', 'SV018_L.csv', 'SV018_R.csv', 'SV019_L.csv', 'SV019_R.csv', 'SV020_L.csv', 'SV020_R.csv', 'SV021_L.csv', 'SV021_R.csv', 'SV022_L.csv', 'SV022_R.csv', 'SV023_L.csv', 'SV023_R.csv', 'SV024_L.csv', 'SV024_R.csv', 'SV025_L.csv', 'SV025_R.csv', 'SV026_L.csv', 'SV026_R.csv', 'SV027_L.csv', 'SV027_R.csv', 'SV028_L.csv', 'SV028_R.csv', 'SV029_L.csv', 'SV029_R.csv', 'SV030_L.csv', 'SV030_R.csv', 'SV031_L.csv', 'SV031_R.csv', 'SV032_L.csv', 'SV032_R.csv', 'SV033_L.csv', 'SV033_R.csv']
Unique subject IDs: ['SV033', 'SV031', 'SV020', 'SV025', 'SV017', 'SV021', 'SV012', 'SV026', 'SV022', 'SV009', 'SV028', 'SV018', 'SV013', 'SV019', 'SV030', 'SV024', 'SV032', 'SV023', 'SV029', 'SV027']
Combined file for subject SV033 already exists. Skipping...
Combined file for subject SV031 already exists. Skipping...
Combined file for subject SV020 already exists. Skipping...
C

### Align eye tracking data with trials  

File naming convention: 

{sid}{eye}_Blink.csv
- eye,tStart,tEnd,duration

{sid}{eye}_Fixation.csv
- eye,tStart,tEnd,duration,xAvg,yAvg,pupilAvg

{sid}{eye}_Message.csv
- time,text
- e.g.
1461608,TRIALID 1  
1461631,FIX_START  
1463138,FIX FAILED  
1464642,FIX_SUCCEED  
1464892,FIX_END  
1464908,STIM_START  
1465159,STIM_END  
1465659,RESP_START  
1482943,RESP_END  
1482943,TRIAL_END 1  

{sid}{eye}_Saccade.csv
- eye,tStart,tEnd,duration,xStart,yStart,xEnd,yEnd,ampDeg,vPeak

{sid}{eye}_Sample.csv
- tSample,LX,LY,LPupil,RX,RY,RPupil

{sid}{eye}_timeStamps.csv
- trialID,fixStart,fixEnd,fixFailed,fixSucceed,stimStart,stimEnd,respStart,respEnd

In [5]:
# Create a dataframe for behavioral data + eye tracking data
# from {sid}.csv
# subj_id,eye,trial_no,n_flash,n_beep,direction,blindspot,response

import os
import pandas as pd
import importlib
import get_df
importlib.reload(get_df)
gdf = get_df.DataFrame()

eyedir = 'edf/asc/ParseEyeLinkAsc'

sids = ['SV009']
for sid in sids:
    try:
        # Load data

        # ------------------------------
        #       Behavioral data
        # ------------------------------
        behavioral_df = pd.read_csv('csv/{}.csv'.format(sid))
        # print(f"Data for subject {sid}:\n", df)

        # ------------------------------
        #       Eyelink Messages
        # ------------------------------
        message_df = gdf.message(sid)        
        trial_event_df = gdf.trial_event(message_df)
        # print(trial_event_df.head())

        # ------------------------------
        #     Saccade data
        # ------------------------------
        saccade_df, saccade_phase_df, saccade_summary = gdf.saccade_by_trial(sid, trial_event_df)
        # print(saccade_phase_df)
        # print(saccade_summary)
        saccade_phase_df.to_csv('test_{}_sac_phase.csv'.format(sid), index=False)

        # ------------------------------
        #    Fixation data
        # ------------------------------
        # fixation_df, fixation_phase_df, fixation_summary = gdf.fixation_by_trial(sid, trial_event_df)
        # fixation_phase_df.to_csv('test_{}_fix_phase.csv'.format(sid), index=False)
        # print(fixation_phase_df)

        # ------------------------------
        #   Combine data
        # ------------------------------
        # phases = ['Fixation', 'Response', 'Stimulus']
        # phases = ['Fixation',  'Stimulus']

        # combined_df = pd.concat(
        #     [behavioral_df.assign(phase=phase) for phase in phases],
        #     ignore_index=True
        # )
        # summary_dfs = [fixation_summary, saccade_summary]
        # for summary_df in summary_dfs:
        #     combined_df = combined_df.merge(summary_df, on=['trial_no', 'eye', 'phase'], how='left')

        # print(combined_df)

        # # Save the combined DataFrame to a new CSV file
        # combined_df.to_csv('behavioral_eyetracking/{}_saccade_fixation_summary.csv'.format(sid), index=False)

    except FileNotFoundError as e:
        print(f"Error: {e}. Skipping subject {sid}.")

     eye   tStart     tEnd  duration    xAvg    yAvg  pupilAvg trial_no  \
0      L  1461865  1462077       216  1923.8  1144.4       706        1   
1      L  1462109  1463021       916  1806.6  1543.9       765        1   
2      L  1463089  1463253       168   357.3  1140.4       795        1   
3      L  1463273  1463389       120   141.3  1183.5       756        1   
4      L  1463453  1464117       668  1903.6  1200.2       808        1   
...   ..      ...      ...       ...     ...     ...       ...      ...   
5562   R  5296030  5296466       440  1949.4   606.5       760      470   
5563   R  5296634  5297006       376  2043.0   953.1       753      470   
5564   R  5300646  5300866       224  1976.6   568.5       595      471   
5565   R  5308626  5309526       904  1895.7  1058.9       773      475   
5566   R  5319966  5320186       224  2001.0   659.0       719      480   

         phase  
0     Fixation  
1     Fixation  
2     Fixation  
3     Fixation  
4     Fixation